# Conv3D Backprop Filter v2 核函数开发

本章以 **Conv3D Backprop Filter v2** 算子为例，完整演示如何在 AscendC 中直接调用卷积反向传播权重梯度核函数。内容涵盖：

- 算子分析：权重梯度数学表达式、输入/输出规格、与前向卷积的核心架构差异。
- 核函数开发：Tiling 结构体、片上缓冲区管理、Load2D(transpose)、Load3D(LEFT Fmatrix)、Mmad、Fixpipe 等关键步骤。
- 主机侧调用：SetTilingData、CheckTilingSpace、ACL 内存管理与核函数启动。

本章学习大纲如下：

- 算子分析：明确 Conv Backprop Filter 的数学表达式、M/K/N 维度映射与前向卷积的差异。
- 核函数开发：逐步实现设备侧权重梯度核函数的各个模块，重点关注 A 侧 Load2D(transpose) 和 B 侧 Load3D(LEFT Fmatrix)。
- 主机侧调用代码：编写 Host 侧驱动程序，完成 Tiling 填充、内存管理与核函数启动。
- 编译与运行：使用 CMake + 毕昇编译器完成编译并执行。

---
## 1. 环境准备

正式开始学习之前，先要对 Jupyter 环境进行初始化。以下代码完成了初始化并将环境变量导入 Jupyter 环境，同时创建代码目录，保证能正常使用毕昇编译器完成算子的开发及编译。

In [ ]:
import os, subprocess
os.makedirs("Sources/01.04", exist_ok=True)
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n Environment initialization process completed successfully!")

## 2. 算子分析

### 2.1 数学表达式

卷积反向传播权重梯度（Conv Backprop Filter）的数学表达式为：

$$dL/dW[c_o, c_i, k_h, k_w] = \sum_{h_o} \sum_{w_o} dL/dY[c_o, h_o, w_o] \times X[c_i, h_o \cdot s_H + k_h \cdot d_H, w_o \cdot s_W + k_w \cdot d_W]$$

即 **dL/dW = dL/dY × im2col(X)**，其中：
- $dL/dY$ 为梯度输出（gradOutput），形状为 `(BATCH, CO, HO, WO)`
- $X$ 为输入特征图（fmap），形状为 `(BATCH, CI, HI, WI)`
- $dL/dW$ 为权重梯度（gradWeight），形状为 `(CO, CI, KH, KW)`

在 Mmad 矩阵乘法视角下，维度映射为：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>维度</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center"><strong>前向 Conv2D</strong></td>
  <td align="center"><strong>Backprop Filter</strong></td>
</tr>
<tr>
  <td align="left">M</td>
  <td align="left">矩阵行数</td>
  <td align="left">HO × WO</td>
  <td align="left"><strong>CO</strong></td>
</tr>
<tr>
  <td align="left">K</td>
  <td align="left">矩阵内积维度</td>
  <td align="left">CI × KH × KW</td>
  <td align="left"><strong>HO × WO</strong></td>
</tr>
<tr>
  <td align="left">N</td>
  <td align="left">矩阵列数</td>
  <td align="left">CO</td>
  <td align="left"><strong>CI × KH × KW</strong></td>
</tr>
</table>

### 2.2 输入与输出

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>张量</strong></td>
  <td align="center"><strong>形状（NCHW）</strong></td>
  <td align="center"><strong>数据类型</strong></td>
  <td align="center"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">梯度输出 dL/dY</td>
  <td align="left">(BATCH, CO, HO, WO)</td>
  <td align="left">fp16</td>
  <td align="left">A 矩阵</td>
</tr>
<tr>
  <td align="left">输入特征图 X</td>
  <td align="left">(BATCH, CI, HI, WI)</td>
  <td align="left">fp16</td>
  <td align="left">B 矩阵</td>
</tr>
<tr>
  <td align="left">权重梯度 dL/dW</td>
  <td align="left">(CO, CI, KH, KW)</td>
  <td align="left">fp16</td>
  <td align="left">输出</td>
</tr>
</table>

输出尺寸推导：

$$HO = \lfloor (HI + pad\_top + pad\_down - d_H(KH-1) - 1) / s_H \rfloor + 1$$

### 2.3 NPU 上存储层次与数据流

Conv Backprop Filter 的数据流与前向卷积存在**关键架构差异**：

```
GM  (Global Memory / HBM)  — 全局存储空间
L1  (L1 缓冲区)             — 每核 512 KB
L0A (矩阵 A 缓冲区)         — 64 KB，送入 Mmad 左操作数
L0B (矩阵 B 缓冲区)         — 64 KB，送入 Mmad 右操作数
L0C (累加器缓冲区)           — 256 KB，存储 Mmad 结果（fp32）
```

**前向 Conv2D 数据流**（参见 01.03）：

```
GM ──DataCopy──► L1  (LoadAL1: fmap, Fmatrix LEFT)
GM ──DataCopy──► L1  (LoadBL1: weight)
L1 ──Load3D──►  L0A  (LoadAL0: im2col 在线展开)
L1 ──Load2D──►  L0B  (LoadBL0: 权重 NZ 布局)
L0A × L0B ──Mmad──► L0C
L0C ──Fixpipe──► GM  (dstStride = HO×WO)
```

**Backprop Filter 数据流**（本章）：

```
GM ──Dn2Nz──► L1  (LoadAL1: gradOutput, 格式转换到 NZ)
GM ──Dn2Nz──► L1  (LoadBL1: fmap, Fmatrix LEFT)
L1 ──Load2D(transpose)──► L0A  (gradOutput NZ → transposed NZ for Mmad)
L1 ──Load3D(im2col)──►   L0B  (fmap im2col on-the-fly, fMatrixCtrl=0 LEFT)
L0A × L0B ──Mmad──► L0C
L0C ──Fixpipe──► GM  (dstStride = CI, COLUMN_MAJOR)
```

核心差异总结：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>差异点</strong></td>
  <td align="center"><strong>前向 Conv2D</strong></td>
  <td align="center"><strong>Backprop Filter</strong></td>
</tr>
<tr>
  <td align="left">A 矩阵来源</td>
  <td align="left">fmap（输入特征图）</td>
  <td align="left"><strong>gradOutput（梯度输出）</strong></td>
</tr>
<tr>
  <td align="left">B 矩阵来源</td>
  <td align="left">weight（权重）</td>
  <td align="left"><strong>fmap（输入特征图）</strong></td>
</tr>
<tr>
  <td align="left">A 侧 L1→L0A</td>
  <td align="left">Load3D（im2col）</td>
  <td align="left"><strong>Load2D（transpose）</strong></td>
</tr>
<tr>
  <td align="left">B 侧 L1→L0B</td>
  <td align="left">Load2D</td>
  <td align="left"><strong>Load3D（im2col, LEFT Fmatrix）</strong></td>
</tr>
<tr>
  <td align="left">A 侧 GM→L1</td>
  <td align="left">DataCopy + Load3DSetFMatrixCal</td>
  <td align="left"><strong>Dn2Nz（无需 Fmatrix）</strong></td>
</tr>
<tr>
  <td align="left">B 侧 Fmatrix</td>
  <td align="left">Load3DSetFMatrixCal（LEFT）</td>
  <td align="left"><strong>Load3DSetFMatrixCal（LEFT）</strong></td>
</tr>
<tr>
  <td align="left">Fixpipe dstStride</td>
  <td align="left">HO × WO</td>
  <td align="left"><strong>orgCi（= CI）</strong></td>
</tr>
</table>

> **Fmatrix 寄存器说明**：dav-3510 上有两个 Fmatrix 寄存器：LEFT（`set_fmatrix`）和 RIGHT（`set_fmatrix_b`）。Load3DBitModeParam 中 `fMatrixCtrl=0` 读取 LEFT 寄存器，`fMatrixCtrl=1` 读取 RIGHT 寄存器。本 Demo 的 B 侧 Load3D 使用 `fMatrixCtrl=0`（LEFT），通过 `Load3DSetFMatrixCal` 设置。

### 2.4 本 Demo 的局限性与约束

本 Demo 是一个**最小可运行实现**，**只支持Ascend 950PR/Ascend 950DT芯片型号**，目的是清晰展示 AscendC 卷积反向传播权重梯度核函数的完整调用链，尤其是与前向卷积的架构差异。为了降低复杂度，做了以下简化：

#### 全载搬运（Full-Load）

`SetTilingData` 中令 `kL0 == kAL1 == kBL1 == HO × WO`，即整个 K 维度在一次 L1→L0 搬运中全部完成，`ddr2l0LoopK` 始终为 1。当 HO×WO 增大时，全载所需的 L0A/L0B 空间会线性增长，很快超出 64 KB 的硬件上限。

#### 单核，无多核分工

`numBlocks = 1`，整个权重梯度由单核完成。没有按 batch、输出通道或空间维度分核的逻辑。

#### 无流水 / 无双缓冲

L1 队列深度为 1（`TQue<..., 1>`），L0 双缓冲关闭（`L0_SYBC_DB_CLOSE`）。数据搬运和矩阵计算串行执行。

#### 数据类型固定为 fp16

输入均为 `half`（fp16），L0C 累加器为 `float`（fp32）。不支持 int8、bf16 等其他数据类型。

#### 数据布局固定为 NCDHW

`ConvFormat` 枚举只定义了 `NCHW = 1`(本demo中D轴为1)。

#### CI 和 CO 均固定为 16

- **CI = 16**：`cinBInCore = orgCi = 16`，每次只处理 16 个输入通道。
- **CO = 16**：`cinAInCore = orgCo = 16`，每次只处理 16 个输出通道。

#### K 维度对齐约束

A 侧 Load2D(transpose) 和 B 侧 Load3D 要求 K = HO×WO 必须是 C0（fp16 下为 16）的倍数。本 Demo 选择 HI=6, WI=6，使得 HO=4, WO=4, K=16 恰好满足此约束。

#### 支持非对称 Padding

与前向 Conv2D Demo（只支持对称 padding）不同，本 Demo 的 `ConvBpFilterTilingData` 包含 `padTop`、`padLeft`、`padRight`、`padDown` 四个独立字段，支持非对称 padding。

#### 空间尺寸上限（由全载搬运决定）

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>缓冲区</strong></td>
  <td align="center"><strong>硬件上限</strong></td>
  <td align="center"><strong>全载用量公式</strong></td>
</tr>
<tr>
  <td align="left">L1</td>
  <td align="left">512 KB / 核</td>
  <td align="left"><code>(al1Space + bl1Space)</code> 字节</td>
</tr>
<tr>
  <td align="left">L0A</td>
  <td align="left">64 KB / 核</td>
  <td align="left"><code>AlignUp(CO, 16) × AlignUp(HO×WO, 16) × 2</code> 字节</td>
</tr>
<tr>
  <td align="left">L0B</td>
  <td align="left">64 KB / 核</td>
  <td align="left"><code>AlignUp(HO×WO, 16) × AlignUp(CI×KH×KW, 16) × 2</code> 字节</td>
</tr>
<tr>
  <td align="left">L0C</td>
  <td align="left">256 KB / 核</td>
  <td align="left"><code>AlignUp(CO, 16) × AlignUp(CI×KH×KW, 16) × 4</code> 字节</td>
</tr>
</table>

> **示例验证**：本 Demo 默认参数（CI=16, HI=WI=6, KH=KW=3, CO=16, 无 padding）下，输入全 1、梯度输出全 1，每个权重梯度元素的计算值为：
>
> $$dL/dW[c_o, c_i, k_h, k_w] = \sum_{h_o=0}^{3} \sum_{w_o=0}^{3} 1 \times 1 = HO \times WO = 4 \times 4 = 16$$
>
> 即每个权重梯度元素值均为 16（= HO × WO）。注意这与前向卷积不同——前向中每个输出点值为 CI×KH×KW=144，而反向中每个权重梯度点值为 HO×WO=16。

---
## 3. 核函数开发

基于对 Conv Backprop Filter V2 算子的分析，我们开始逐步实现核函数代码。所有代码均写入 `Sources/01.04/minimal_demo.asc`，采用增量追加方式。

### 3.1 头文件与 Tiling 结构体

已在下方代码单元格中完成。**ConvBpFilterTilingData** 保存了核函数所需的全部参数。与前向 Conv2D 的 `Conv2DTilingData` 相比，主要差异：

- **新增字段**：`alignedL1UseKa`（A 侧 L1 NZ 格式对齐 K）、`kStep`（Load2D kStep shifted 值）、`bL1SpaceSize`（B 侧 L1 缓冲区大小）、`padRight` / `padDown`（支持非对称 padding）。
- **移除字段**：`nL1DivBlockSize`（原为 Load2D srcStride，已不再需要）、`hoL0` / `woL0`（前向的空间分块）。



In [ ]:
%%writefile Sources/01.04/minimal_demo.asc
/**
 * Copyright (c) 2025 Huawei Technologies Co., Ltd.
 */

/*!
 * \file conv3d_backprop_filter_v2_demo.asc
 * \brief Standalone demo for direct-calling Conv3D Backprop Filter v2 kernel.
 *
 * ============================================================
 * Tutorial overview
 * ============================================================
 * This file is the host-side driver for the Conv3D Backprop Filter v2 kernel demo.
 * It covers the full lifecycle of an AscendC kernel call for the weight-gradient
 * (dL/dW) computation in convolution backward pass.
 *
 *   1. Define conv shape constants (BATCH, CI, HI, WI, CO, KH, KW, …)
 *   2. Fill in the tiling struct (SetTilingData) — tells the kernel how
 *      to partition work across the NPU memory hierarchy.
 *   3. Validate that the tiling fits in on-chip SRAM (CheckTilingSpace).
 *   4. Allocate device memory, copy inputs, launch the kernel.
 *   5. Copy results back and dump to text files for inspection.
 *
 * ============================================================
 * Key difference from Conv2D (forward)
 * ============================================================
 *
 * Conv2D (forward) — A matrix Load3D, B matrix Load2D:
 *   A matrix (fmap)  ──Load3D──► L0A   (im2col on-the-fly)
 *   B matrix (weight)──Load2D──► L0B   (NZ layout)
 *   Mmad: L0A × L0B ──► L0C
 *   M = HO × WO,  K = CI × KH × KW,  N = CO
 *
 * Conv3D Backprop Filter v2 — A matrix Load2D(transpose), B matrix Load3D(im2col):
 *   A matrix (gradOutput/dL/dY) ──Load2D(transpose)──► L0A  (NZ format + transpose)
 *   B matrix (fmap/X)           ──Load3D(im2col)──►   L0B  (im2col on-the-fly, fMatrixCtrl=0 LEFT)
 *   Mmad: L0A × L0B ──► L0C
 *   M = CO,  K = HO × WO,  N = CI × KH × KW
 *
 * The mathematical formula: dL/dW = dL/dY × im2col(X)
 *   - dL/dY (gradOutput) acts as left matrix A, loaded via Load2D with transpose
 *   - X (fmap/Input) goes through im2col and acts as right matrix B, loaded via Load3D
 *   - dL/dW (gradWeight) is the output C
 *
 * ============================================================
 * NPU memory hierarchy (smallest / fastest → largest / slowest):
 *
 *   GM  (Global Memory / HBM)  — off-chip, GBs, ~100s GB/s
 *   L1  (on-chip SRAM)         — 512 KB per core, ~TB/s
 *   L0A (Matrix A buffer)     — 64 KB, feeds left operand of Mmad
 *   L0B (Matrix B buffer)     — 64 KB, feeds right operand of Mmad
 *   L0C (Accumulator buffer)  — 256 KB, stores Mmad result (fp32)
 *
 * Data flow for a single Conv Backprop Filter tile:
 *
 *   GM ──Dn2Nz──► L1  (LoadAL1: gradOutput, format conversion to NZ)
 *   GM ──Dn2Nz──► L1  (LoadBL1: fmap, format conversion to NZ)
 *   L1 ──Load2D(transpose)──► L0A  (LoadAL0: gradOutput NZ → transposed NZ for Mmad)
 *   L1 ──Load3D(im2col)──►   L0B  (LoadBL0: fmap im2col on-the-fly, Fmatrix LEFT)
 *   L0A × L0B ──Mmad──► L0C
 *   L0C ──Fixpipe──► GM  (fp32→fp16 cast + weight layout)
 *
 * Fmatrix register mapping (dav-3510 specific):
 *   Load3DSetFMatrixCal  → set_fmatrix   (LEFT register)
 *   Load3DSetFMatrixBCal → set_fmatrix_b (RIGHT register)
 *   In Load3DBitModeParam: fMatrixCtrl=0 reads LEFT, fMatrixCtrl=1 reads RIGHT
 *
 *   In this backprop filter demo, the B-side Load3D uses fMatrixCtrl=0 (LEFT),
 *   set via Load3DSetFMatrixCal. This differs from some forward conv implementations
 *   where B-side uses fMatrixCtrl=1 (RIGHT) via Load3DSetFMatrixBCal.
 * ============================================================
 */

#include <cstdint>
#include <cstring>
#include <iostream>
#include <fstream>
#include <cmath>
#include <limits>
#include "acl/acl.h"
#include "kernel_operator.h"

// ============================================================================
// Tiling data structure
// ============================================================================
//
// ConvBpFilterTilingData holds all parameters the kernel needs for the
// backprop filter computation. The M/K/N dimensions are:
//   M = CO (output channels from gradOutput)
//   K = HO × WO (spatial reduction dimension)
//   N = CI × KH × KW (input channels × kernel spatial from fmap im2col)

// pack(1) is required: this struct is copied verbatim from Host to Device GM as a
// contiguous byte stream for kernel tiling consumption. Any padding bytes would
// corrupt the tiling data layout expected by the aicore kernel.
#pragma pack(1)

struct ConvBpFilterTilingData {
    uint64_t orgHi = 0;
    uint64_t orgWi = 0;
    uint64_t orgHo = 0;
    uint64_t orgWo = 0;
    uint64_t singleCoreBatch = 0;
    uint32_t orgCi = 0;
    uint32_t orgCo = 0;
    uint32_t singleCoreM = 0;
    uint32_t singleCoreK = 0;
    uint32_t singleCoreN = 0;
    uint32_t kAL1 = 0;
    uint32_t kBL1 = 0;
    uint32_t nBL1 = 0;
    uint32_t mStep = 0;
    uint32_t kL0 = 0;
    uint32_t nL0 = 0;
    uint32_t kStep = 0;
    uint32_t alignedL1UseKa = 0;
    uint32_t orgHixWi = 0;
    uint32_t kernelHxkernelW = 0;
    uint32_t aL1SpaceSize = 0;
    uint32_t bL1SpaceSize = 0;
    uint32_t cinAInCore = 0;
    uint32_t cinBInCore = 0;
    uint32_t channelSize = 0;
    uint32_t kernelH = 0;
    uint32_t kernelW = 0;
    uint32_t strideH = 0;
    uint32_t strideW = 0;
    uint32_t dilationH = 0;
    uint32_t dilationW = 0;
    int8_t offsetx = 0;
    uint32_t padTop = 0;
    uint32_t padLeft = 0;
    uint32_t padRight = 0;
    uint32_t padDown = 0;
};

#pragma pack()

__aicore__ inline void CopyConvBpFilterTiling(ConvBpFilterTilingData* dst, GM_ADDR tilingGM)
{
    uint32_t* ptr = reinterpret_cast<uint32_t*>(dst);
    auto src = reinterpret_cast<__gm__ uint32_t*>(tilingGM);
    for (uint32_t i = 0; i < sizeof(ConvBpFilterTilingData) / sizeof(uint32_t); i++, ptr++) {
        *ptr = *(src + i);
    }
}

#define GET_TILING_DATA(tilingData, tilingArg) \
    ConvBpFilterTilingData tilingData;               \
    CopyConvBpFilterTiling(&tilingData, tilingArg)

### 3.2 NPU存储层次与常量

`#ifdef __CCE_AICORE__` 块内的代码只在设备侧编译。`namespace conv_bp_filter` 中定义了缓冲区大小常量、Load3D/Load2D 位域偏移量以及辅助函数。

**与前向 Conv2D 的关键差异**：

- **新增位域偏移**：`STRIDEW_OFFSET`、`FILTERW_OFFSET`、`FILTERH_OFFSET`、`FILTERSIZEW_OFFSET`、`FILTERSIZEH_OFFSET`、`TRANSPOSE_OFFSET`、`FMATRIXCTRL_OFFSET`、`CHANNELSIZE_OFFSET`。这些偏移用于构造 Load3DBitModeParam 的 config1 位域。
- **新增辅助函数**：`ShiftCeilDiv16`、`ShiftDiv16`。Load2D 的 kStep/mStep/srcStride/dstStride 使用"shifted 单位"（每单位=16 个元素），这些函数用于计算 shifted 值。`ShiftCeilDiv16(a)` 计算 CeilDiv(a, 16) 的 shifted 值，`ShiftDiv16(a)` 计算 a/16 的 shifted 值。
- **修正偏移**：`FILTERSIZEW_OFFSET = 44` 和 `FILTERSIZEH_OFFSET = 45`（而非前向 Demo 中的 36/37，后者是错误的）。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
#ifdef __CCE_AICORE__
#include "kernel_basic_intf.h"
#include "kernel_tiling/kernel_tiling.h"

using namespace AscendC;

namespace conv_bp_filter {

constexpr uint64_t L0A_SIZE = 65536;
constexpr uint64_t L0B_SIZE = 65536;
constexpr uint64_t L0C_SIZE = 262144;
constexpr uint64_t C0_SIZE = 32;
constexpr uint64_t PAD_IDX_T = 2;
constexpr uint64_t PAD_IDX_L = 0;
constexpr uint64_t PAD_IDX_R = 1;
constexpr uint64_t MAX_PAD_R = 255;
constexpr uint64_t MIN_HI_WI = 1;
constexpr uint32_t BLOCK_L0_N = 16;
constexpr uint32_t BLOCK_L0_M = 16;
constexpr uint32_t L0_SYBC_DB_CLOSE = 0x0;

constexpr uint8_t MSTEP_OFFSET = 16;
constexpr uint8_t POSM_OFFSET = 48;
constexpr uint8_t POSK_OFFSET = 32;

// Bit offsets for Load3DBitModeParam config1 (used for B-side Load3D)
// Layout (from lowest bit to highest):
//   strideW(6) | strideH(6) | filterW(8) | filterH(8)
//   | dilationW(8) | dilationH(8) | filterSizeW(1) | filterSizeH(1)
//   | transpose(1) | fMatrixCtrl(1) | channelSize(16)
constexpr uint8_t STRIDEW_OFFSET = 0;
constexpr uint8_t STRIDEH_OFFSET = 6;
constexpr uint8_t FILTERW_OFFSET = 12;
constexpr uint8_t FILTERH_OFFSET = 20;
constexpr uint8_t DILATIONW_OFFSET = 28;
constexpr uint8_t DILATIONH_OFFSET = 36;
constexpr uint8_t FILTERSIZEW_OFFSET = 44;
constexpr uint8_t FILTERSIZEH_OFFSET = 45;
constexpr uint8_t TRANSPOSE_OFFSET = 46;
constexpr uint8_t FMATRIXCTRL_OFFSET = 47;
constexpr uint8_t CHANNELSIZE_OFFSET = 48;

constexpr uint64_t MASK_16 = 0xffff;
constexpr uint64_t MASK_8 = 0xff;
constexpr uint64_t MASK_6 = 0x3f;
constexpr uint64_t MASK_1 = 0x1;
constexpr uint64_t NINTH_BIT_MASK = 0x100;

// 0x3F800000 = bit pattern of IEEE-754 float32 1.0, used as dequantization scalar
constexpr uint64_t DEQ_SCALAR_ONE = 0x3F800000UL;


constexpr uint8_t UNIT_FLAG_ENABLE_ONLY = 2;
constexpr uint8_t UNIT_FLAG_ENABLE_WITH_FLIP = 3;

static __aicore__ inline uint64_t AlignB(uint64_t a, uint64_t b)
{
    return ((a + b - 1) / b) * b;
}

static __aicore__ inline uint64_t CeilDiv(uint64_t a, uint64_t b)
{
    return (a + b - 1) / b;
}

// CeilDiv(a, 16) in shifted units — used for Load2D kStep/mStep and NZ format stride
static __aicore__ inline uint64_t ShiftCeilDiv16(uint64_t a)
{
    return (a + 15) >> 4;
}

// a / 16 in shifted units — used for Load2D srcStride
static __aicore__ inline uint64_t ShiftDiv16(uint64_t a)
{
    return a >> 4;
}

### 3.3 数据类型定义

本节定义三个核心类型：

- **ConvFormat**：枚举卷积数据布局，当前仅支持 NCHW。
- **ConvType**：将存储位置（TPosition）、数据布局（ConvFormat）和元素类型（TYPE）打包为一个模板类型标签。
- **ConvBpFilterContext**：保存单次权重梯度迭代所需的全部状态。与前向 `Conv2dContext` 的关键差异：
  - 只有 `SRC_TYPE` 和 `DST_TYPE`（无 WEIGHT_TYPE），因为反向传播中 A/B 矩阵均为同类型数据（fp16）。
  - `doutGm` 和 `fmapGm`（替代前向的 `agm` 和 `bgm`）。
  - `alignedL1UseKa_`：A 侧 L1 NZ 格式的对齐 K 维度，用于 Load2D srcStride 计算。
  - 无缓存 tiling 字段（如 `dilatedKernelH/W`、`orgCi/Co` 等），这些通过 `ctx.convTiling->xxx` 直接访问。
  - 无 `hiStartPos` / `wiStartPos`（反向中 padding 由 LoadBL1 从几何关系推算）。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
enum class ConvFormat : std::uint8_t {
    NCHW = 1,
};

template <TPosition POSITION, ConvFormat FORMAT, typename TYPE>
struct ConvType {
    using T = TYPE;
};

template <class SRC_TYPE, class DST_TYPE>
struct ConvBpFilterContext {
    using SrcT = typename SRC_TYPE::T;
    using DstT = typename DST_TYPE::T;
    using L0cT = float;

    constexpr static uint64_t k0 = C0_SIZE / sizeof(SrcT);

    TPipe pipe;
    GlobalTensor<SrcT> doutGm;
    GlobalTensor<SrcT> fmapGm;
    TBuf<TPosition::A2> al0Buf;
    TBuf<TPosition::B2> bl0Buf;
    TBuf<TPosition::CO1> l0cBuf;

    LocalTensor<SrcT> al1;
    LocalTensor<SrcT> bl1;
    LocalTensor<SrcT> wholeAl0Tensor;
    LocalTensor<SrcT> al0;
    LocalTensor<SrcT> wholeBl0Tensor;
    LocalTensor<SrcT> bl0;
    LocalTensor<L0cT> wholeCl0Tensor = LocalTensor<L0cT>(TPosition::CO1, 0, 0);
    LocalTensor<L0cT> cl0 = LocalTensor<L0cT>(TPosition::CO1, 0, 0);

    TQue<QuePosition::A1, 1> queueAL1;
    TQue<QuePosition::B1, 1> queueBL1;

    const ConvBpFilterTilingData* __restrict convTiling = nullptr;

    uint64_t currentML0 = 0;
    uint64_t currentKL0 = 0;
    uint64_t currentNL0 = 0;
    uint64_t currentML0Align = 0;
    uint64_t currentNL0Align = 0;

    uint64_t ddr2l0LoopK = 0;
    uint64_t maxKL0Iter = 0;
    uint64_t kL0Tail = 0;
    uint64_t kIter = 0;
    uint64_t kAL0Iter = 0;

    uint64_t alignedL1UseKa_ = 0;

    uint32_t bL1SpaceSize = 0;
};

### 3.4 上下文初始化 InitContext

`InitContext` 从 Tiling 结构体中读取所有形状参数，初始化片上缓冲区和 L1 队列，并预计算循环控制变量。

关键逻辑：
- **convTiling 指针**：所有 tiling 参数通过 `ctx.convTiling->xxx` 直接访问，不再缓存到上下文字段。
- **alignedL1UseKa_**：A 侧 L1 NZ 格式的对齐 K 维度 = ShiftCeilDiv16(kAL1) × k0。用于 Load2D 的 srcStride 计算。
- **ddr2l0LoopK / kL0Tail**：K 维度的循环次数和尾块大小。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
template <class Ctx>
__aicore__ inline void InitContext(Ctx &ctx, const ConvBpFilterTilingData *tiling)
{
    ctx.convTiling = tiling;

    if ASCEND_IS_AIC {
        ctx.currentML0 = tiling->singleCoreM;
        ctx.currentKL0 = tiling->singleCoreK;
        ctx.currentNL0 = tiling->nBL1;

        ctx.pipe.InitBuffer(ctx.al0Buf, L0A_SIZE);
        ctx.pipe.InitBuffer(ctx.bl0Buf, L0B_SIZE);
        ctx.pipe.InitBuffer(ctx.l0cBuf, L0C_SIZE);
        ctx.wholeAl0Tensor = ctx.al0Buf.template Get<typename Ctx::SrcT>();
        ctx.wholeBl0Tensor = ctx.bl0Buf.template Get<typename Ctx::SrcT>();
        ctx.wholeCl0Tensor = ctx.l0cBuf.template Get<typename Ctx::L0cT>();
        ctx.bL1SpaceSize = tiling->nBL1 * tiling->kBL1;
        ctx.pipe.InitBuffer(ctx.queueAL1, 1, tiling->aL1SpaceSize);
        ctx.pipe.InitBuffer(ctx.queueBL1, 1, ctx.bL1SpaceSize * sizeof(typename Ctx::SrcT));

        ctx.alignedL1UseKa_ = ShiftCeilDiv16(tiling->kAL1) * Ctx::k0;

        uint64_t totalK = tiling->singleCoreK;
        ctx.ddr2l0LoopK = CeilDiv(totalK, tiling->kL0);
        ctx.maxKL0Iter = ctx.ddr2l0LoopK - 1;
        ctx.kL0Tail = totalK % tiling->kL0;
        ctx.kL0Tail = ctx.kL0Tail == 0 ? tiling->kL0 : ctx.kL0Tail;

        ctx.currentML0Align = tiling->mStep;
        ctx.currentNL0Align = tiling->nL0;
    }
}

### 3.5 数据加载：LoadAL1 / LoadBL1

**LoadAL1**：将梯度输出（gradOutput/dL/dY）从 GM 搬运到 L1。

核心差异：前向 Conv2D 中 A 侧使用 `Load3DSetFMatrixCal` + `DataCopy`（行主序），而反向中 A 侧使用 `Dn2NzParams`（NZ 格式转换在 GM→L1 阶段完成），**不调用 Load3DSetFMatrixCal**，因为 A 侧使用 Load2D（而非 Load3D）。

**LoadBL1**：将输入特征图（fmap/X）从 GM 搬运到 L1。

核心差异：前向 Conv2D 中 B 侧直接搬运权重，而反向中 B 侧搬运 fmap 并设置 Fmatrix。关键点是使用 **Load3DSetFMatrixCal（LEFT 寄存器）** 而非 `Load3DSetFMatrixBCal（RIGHT 寄存器）`。

#### Dn2NzParams详解

**功能**：将NCHW格式转换为NZ格式。在卷积算子实现中，DataCopy不仅完成数据搬运，还承担着格式转换的关键角色。Cube单元要求输入数据为特定的分形格式（Fractal Format），因此需要在数据从GM搬运至L1的过程中完成格式转换。

**Dn2NzParams结构体参数定义**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
</tr>
<tr>
  <td align="left">dnNum</td>
  <td align="left">传输DN矩阵的数目。</td>
  <td align="left">[0, 4095]</td>
</tr>
<tr>
  <td align="left">nValue</td>
  <td align="left">DN矩阵的行数（矩阵高度）。</td>
  <td align="left">[0, 16384]</td>
</tr>
<tr>
  <td align="left">dValue</td>
  <td align="left">DN矩阵的列数（矩阵宽度）。当dValue不满足32B对齐时，目的操作数中不足部分会被补齐为0。</td>
  <td align="left">[0, 2^32-1]</td>
</tr>
<tr>
  <td align="left">srcDnMatrixStride</td>
  <td align="left">源操作数相邻DN矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[0, 2^64-1]</td>
</tr>
<tr>
  <td align="left">srcDValue</td>
  <td align="left">源操作数同一DN矩阵的相邻行起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 2^64-1]</td>
</tr>
<tr>
  <td align="left">dstNzC0Stride</td>
  <td align="left">DN转换到NZ格式后，源操作数中的一列会转换为目的操作数的多行。dstNzC0Stride表示目的NZ矩阵中，来自源操作数同一列的多行数据相邻行起始地址间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 65535]</td>
</tr>
<tr>
  <td align="left">dstNzNStride</td>
  <td align="left">目的NZ矩阵中，Z型矩阵相邻行起始地址之间的偏移，单位：C0_SIZE（32B）。</td>
  <td align="left">[1, 65535]</td>
</tr>
<tr>
  <td align="left">dstNzMatrixStride</td>
  <td align="left">目的NZ矩阵中，相邻NZ矩阵起始地址间的偏移，单位：元素。</td>
  <td align="left">[1, 2^32-1]</td>
</tr>
</table>

详细的DN2NZ转换示意图请参考[昇腾社区官方文档](https://www.hiascend.com/document/detail/zh/canncommercial/900/API/ascendcopapi/atlasascendc_api_07_00127.html)。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
// ============================================================================
// LoadAL1: gradOutput (dL/dY) from GM to L1 using Dn2Nz
// ============================================================================
// In backprop filter, the A matrix is gradOutput (dL/dY).
// Key difference from the old (Load3D) approach:
//   Old:  A (gradOutput) → L1 using plain DataCopy (row-major), then
//         L1 → L0A via Load3D (transpose, format conversion at L1→L0A)
//   New:  A (gradOutput) → L1 using Dn2Nz (format conversion at GM→L1),
//         then L1 → L0A via Load2D (transpose only, data already in NZ format)
//
// The Dn2Nz converts gradOutput from NCDHW row-major to NZ format in L1.
// For fp16 with baseUseM_ != 1 and not splitWo:
//   dnNum = 1
//   nValue = kal1_ = HO*WO (K per A1 tile)
//   dValue = curStepM * baseM = CO (M dimension)
//   srcDValue = hwO_ = HO*WO (source stride in NCDHW format)
//   dstNzC0Stride = CeilDiv(nValue, k0) * k0 (NZ column stride)
//
// Since A-side now uses Load2D (not Load3D), we no longer call
// Load3DSetFMatrixCal or Load3DSetPaddingCal for the A-side.
template <class Ctx>
__aicore__ inline void LoadAL1(Ctx &ctx)
{
    ctx.al1 = ctx.queueAL1.template AllocTensor<typename Ctx::SrcT>();

    const auto *t = ctx.convTiling;

    // Dn2Nz params for fp16, not splitWo (from production LoadToA1Fp16Dn2Nz)
    Dn2NzParams dn2NzParams;
    dn2NzParams.dnNum = 1;                                          // not splitWo
    dn2NzParams.nValue = t->kAL1;                                   // K per A1 tile = HO*WO
    dn2NzParams.dValue = t->singleCoreM;                            // M = CO
    dn2NzParams.srcDValue = t->orgHo * t->orgWo;                    // hwO (NCDHW spatial stride)
    dn2NzParams.dstNzC0Stride = ShiftCeilDiv16(t->kAL1) * Ctx::k0;
    dn2NzParams.dstNzNStride = 1;                                   // NZ row stride
    dn2NzParams.dstNzMatrixStride = 1;                              // dnNum=1, matrix stride = 1

    DataCopy<typename Ctx::SrcT>(ctx.al1, ctx.doutGm[0], dn2NzParams);
    ctx.queueAL1.EnQue(ctx.al1);
}

// ============================================================================
// LoadBL1: fmap (X) from GM to L1
// ============================================================================
// In backprop filter, the B matrix is the input feature map (fmap).
// It goes through im2col via Load3D on the B-side (L0B).
//
// Key difference from the old approach:
//   Old:  Load3DSetFMatrixBCal (RIGHT register) for B-side fmap
//   New:  Load3DSetFMatrixCal (LEFT register) for B-side fmap
//
// The Fmatrix for B-side is set via Load3DSetFMatrixCal (set_fmatrix, LEFT register).
// In Load3DBitModeParam, fMatrixCtrl=0 reads the LEFT register.
// This is because in the production backprop filter code, the B-side Load3D
// uses the LEFT register, not the RIGHT register.
template <class Ctx>
__aicore__ inline void LoadBL1(Ctx &ctx)
{
    ctx.bl1 = ctx.queueBL1.template AllocTensor<typename Ctx::SrcT>();

    const auto *t = ctx.convTiling;
    uint64_t dilatedKernelH = 1 + (t->kernelH - 1) * t->dilationH;
    uint64_t dilatedKernelW = 1 + (t->kernelW - 1) * t->dilationW;
    int64_t paddingHi = (t->orgHo - 1) * t->strideH + dilatedKernelH;
    int64_t paddingWi = (t->orgWo - 1) * t->strideW + dilatedKernelW;

    int64_t hiTop = -static_cast<int64_t>(t->padTop);
    int64_t wiLeft = -static_cast<int64_t>(t->padLeft);
    int64_t hiBottomRaw = hiTop + paddingHi;
    int64_t wiRightRaw = wiLeft + paddingWi;

    uint64_t padTopL1 = hiTop < 0 ? uint64_t(-hiTop) : 0;
    // When hiTop > 0 (positive offset from padding), the effective bottom boundary
    // shifts up by hiTop, so we subtract hiTop to get the actual overlap with orgHi.
    int64_t effectiveHiBottom = hiTop > 0 ? hiBottomRaw - hiTop : hiBottomRaw;
    uint64_t padBottomL1 = effectiveHiBottom > (int64_t)t->orgHi ? uint64_t(effectiveHiBottom - t->orgHi) : 0;
    uint64_t padLeftL1 = wiLeft < 0 ? uint64_t(-wiLeft) : 0;
    int64_t effectiveWiRight = wiLeft > 0 ? wiRightRaw - wiLeft : wiRightRaw;
    uint64_t padRightL1 = effectiveWiRight > (int64_t)t->orgWi ? uint64_t(effectiveWiRight - t->orgWi) : 0;

    bool allPad = (padTopL1 >= (uint64_t)paddingHi || padBottomL1 >= (uint64_t)paddingHi ||
                   padLeftL1 >= (uint64_t)paddingWi || padRightL1 >= (uint64_t)paddingWi);
    uint64_t hiLoad = 0, wiLoad = 0;
    if (!allPad) {
        hiLoad = paddingHi - padTopL1 - padBottomL1;
        hiLoad = hiLoad > t->orgHi ? t->orgHi : hiLoad;
        wiLoad = paddingWi - padLeftL1 - padRightL1;
    }

    // B-side Fmatrix uses LEFT register (Load3DSetFMatrixCal)
    // This differs from the old approach which used RIGHT register (Load3DSetFMatrixBCal)
    uint8_t padList[PAD_SIZE] = {MAX_PAD_R, MAX_PAD_R, MAX_PAD_R, MAX_PAD_R};
    if (unlikely(allPad)) {
        Load3DSetFMatrixCal(MIN_HI_WI, MIN_HI_WI, padList);
    } else {
        padList[PAD_IDX_L] = padLeftL1;
        padList[PAD_IDX_R] = padRightL1;
        padList[PAD_IDX_T] = padTopL1;
        Load3DSetFMatrixCal(hiLoad, wiLoad, padList);
    }
    Load3DSetPaddingCal(t->offsetx);

    if (allPad) {
        InitConstValueParams<typename Ctx::SrcT> params(
            1, static_cast<uint16_t>(t->bL1SpaceSize / C0_SIZE), 0, 0);
        InitConstValue<typename Ctx::SrcT>(ctx.bl1, params);
        ctx.queueBL1.EnQue(ctx.bl1);
        return;
    }

    int64_t realHiTopGm = hiTop < 0 ? 0 : hiTop;
    int64_t realWiTopGm = wiLeft < 0 ? 0 : wiLeft;
    int64_t bL1GmOffset = realHiTopGm * t->orgWi + realWiTopGm;

    Dn2NzParams params;
    if (likely(wiLoad == t->orgWi)) {
        params.dnNum = 1;
        params.nValue = hiLoad * wiLoad;
        params.srcDnMatrixStride = 0;
        params.dstNzMatrixStride = 0;
    } else {
        params.dnNum = hiLoad;
        params.nValue = wiLoad;
        params.srcDnMatrixStride = t->orgWi;
        params.dstNzMatrixStride = wiLoad * Ctx::k0;
    }
    params.dValue = t->cinBInCore;
    params.srcDValue = t->orgHixWi;
    params.dstNzC0Stride = hiLoad * wiLoad;
    params.dstNzNStride = 1;
    DataCopy<typename Ctx::SrcT>(ctx.bl1, ctx.fmapGm[bL1GmOffset], params);
    ctx.queueBL1.EnQue(ctx.bl1);
}

### 3.6 数据搬运与 im2col：LoadAL0 / LoadBL0

**LoadAL0**：将 L1 中的 gradOutput 通过 **Load2D(transpose)** 搬运到 L0A。

这是与前向卷积最核心的差异：前向中 A 侧使用 Load3D（im2col），而反向中 A 侧使用 Load2D（transpose）。因为 gradOutput 在 L1 中已经是 NZ 格式（由 Dn2Nz 转换），只需通过 Load2D 的 `ifTranspose=true` 将其转置，使 stored_K→actual_M(=CO), stored_M→actual_K(=HO×WO)。

Load2D 使用 **shifted 单位**：kStep/mStep/srcStride/dstStride 的每单位 = 16 个元素。例如 kStep=1（shifted）表示实际 16 个元素。

#### Load2D（BitMode）详解

**功能定位**：Load2D用于将数据从L1搬运至L0A或L0B，支持矩阵转置和分块搬运。

**LoadData2DParamsV2 接口**（本章使用）：

```cpp
LoadData2DParamsV2 load2dParamsV2;
load2dParamsV2.mStartPosition = 0;     // M 起始位置
load2dParamsV2.kStartPosition = 0;     // K 起始位置
load2dParamsV2.kStep = ...;            // K 步长（shifted 单位，每单位=16元素）
load2dParamsV2.mStep = ...;            // M 步长（shifted 单位）
load2dParamsV2.srcStride = ...;        // 源步长（shifted 单位）
load2dParamsV2.dstStride = ...;        // 目的步长（shifted 单位）
load2dParamsV2.ifTranspose = true;     // 是否转置

Load2DBitModeParam param(load2dParamsV2);
LoadData<TPosition::A2, TPosition::A1, half>(al0, al1, param);
```

**LoadData2DParamsV2 结构体参数**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center" style="width: 150px"><strong>参数名称</strong></td>
  <td align="center"><strong>含义</strong></td>
  <td align="center" style="width: 180px"><strong>取值范围</strong></td>
</tr>
<tr>
  <td align="left">mStartPosition</td>
  <td align="left">M 维度起始位置。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">kStartPosition</td>
  <td align="left">K 维度起始位置。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">kStep</td>
  <td align="left">K 维度步长（shifted 单位，每单位=16元素）。转置后对应 Mmad 的 M 维度。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">mStep</td>
  <td align="left">M 维度步长（shifted 单位）。转置后对应 Mmad 的 K 维度。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">srcStride</td>
  <td align="left">源操作数步长（shifted 单位）。= alignedL1UseKa / k0。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">dstStride</td>
  <td align="left">目的操作数步长（shifted 单位）。= kStep。</td>
  <td align="left">[0, 65535]</td>
</tr>
<tr>
  <td align="left">ifTranspose</td>
  <td align="left">是否启用转置。转置使 stored_K→actual_M, stored_M→actual_K。</td>
  <td align="left">true/false</td>
</tr>
</table>

**LoadBL0**：将 L1 中的 fmap 通过 **Load3D(im2col, LEFT Fmatrix)** 搬运到 L0B。

关键差异：`fMatrixCtrl=0`（LEFT 寄存器），而非前向 Conv2D 中 A 侧使用的 `fMatrixCtrl` 配置。B 侧 Load3D 执行 im2col 在线展开，使用实际卷积参数（步长、核尺寸、膨胀）。

#### Load3D详解

**功能说明**：Load3D用于完成image to column操作，将多维feature map转为二维矩阵。支持通路：A1→A2; B1→B2。

**Load3DBitModeParam config1 位域布局**（本章使用）：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>位段</strong></td>
  <td align="center"><strong>位偏移</strong></td>
  <td align="center"><strong>含义</strong></td>
</tr>
<tr>
  <td align="left">strideW</td>
  <td align="left">[0:5]</td>
  <td align="left">卷积步长 W</td>
</tr>
<tr>
  <td align="left">strideH</td>
  <td align="left">[6:11]</td>
  <td align="left">卷积步长 H</td>
</tr>
<tr>
  <td align="left">filterW</td>
  <td align="left">[12:19]</td>
  <td align="left">卷积核 W</td>
</tr>
<tr>
  <td align="left">filterH</td>
  <td align="left">[20:27]</td>
  <td align="left">卷积核 H</td>
</tr>
<tr>
  <td align="left">dilationW</td>
  <td align="left">[28:35]</td>
  <td align="left">膨胀 W</td>
</tr>
<tr>
  <td align="left">dilationH</td>
  <td align="left">[36:43]</td>
  <td align="left">膨胀 H</td>
</tr>
<tr>
  <td align="left">filterSizeW</td>
  <td align="left">[44]</td>
  <td align="left">filterW > 255 时置 1</td>
</tr>
<tr>
  <td align="left">filterSizeH</td>
  <td align="left">[45]</td>
  <td align="left">filterH > 255 时置 1</td>
</tr>
<tr>
  <td align="left">transpose</td>
  <td align="left">[46]</td>
  <td align="left">转置标志</td>
</tr>
<tr>
  <td align="left">fMatrixCtrl</td>
  <td align="left">[47]</td>
  <td align="left"><strong>Fmatrix 寄存器选择：0=LEFT, 1=RIGHT</strong></td>
</tr>
<tr>
  <td align="left">channelSize</td>
  <td align="left">[48:63]</td>
  <td align="left">通道大小</td>
</tr>
</table>

> **注意**：本 Demo 中 B 侧 Load3D 的 `fMatrixCtrl=0`（LEFT 寄存器），通过 `Load3DSetFMatrixCal` 设置。这与某些旧版实现使用 RIGHT 寄存器不同。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
// ============================================================================
// LoadAL0: gradOutput from L1 to L0A via Load2D (transpose mode)
// ============================================================================
// Key difference from the old approach:
//   Old:  A (gradOutput) → L0A via Load3D (transpose=true, fMatrixCtrl=0 LEFT)
//         This required Load3DSetFMatrixCal in LoadAL1 and Load3DBitModeParam
//   New:  A (gradOutput) → L0A via Load2D (transpose=true)
//         L1 data is already in NZ format (from Dn2Nz), Load2D just transposes
//
// For fp16 with baseUseM_ != 1 (from production CalcParamsL12L0a + LoadL12L0a):
//   kStep = CeilDiv(baseUseM, m0) = CeilDiv(CO, 16) = 1 (shifted units of 16)
//         → actual kStep = 1 * 16 = 16 elements (Mmad M dimension)
//   dstStride = kStep = 1 (shifted) → actual = 16
//   srcStride = alignedL1UseKa / k0 = 16 / 16 = 1 (shifted) → actual = 16
//   mStep = CeilDiv(baseUseK, k0) = CeilDiv(HO*WO, 16) = 1 (shifted) → actual = 16
//   ifTranspose = true
//
// With transpose, the Cube engine reads L0A such that:
//   stored_K (= kStep direction) → actual_M = CO ✓
//   stored_M (= mStep direction) → actual_K = HO*WO ✓
//
// No SetLoadDataRepeat needed for Load2D (repeat is handled differently).
template <class Ctx>
__aicore__ inline void LoadAL0(Ctx &ctx, bool isFirst)
{
    if ASCEND_IS_AIV {
        return;
    }
    const auto *t = ctx.convTiling;
    uint64_t currentKL0 = (ctx.kIter == ctx.maxKL0Iter) ? ctx.kL0Tail : t->kL0;

    // Construct Load2D params from production code convention
    // For fp16 with baseUseM_ != 1, not splitWo, with transpose:
    LoadData2DParamsV2 load2dParamsV2;
    load2dParamsV2.mStartPosition = 0;     // single M tile, start at 0
    load2dParamsV2.kStartPosition = 0;     // single K tile, start at 0
    // kStep = CeilDiv(baseUseM_, m0) in shifted units (= 1 for CO=16)
    // In actual elements: 1 * 16 = 16. We store the shifted value.
    load2dParamsV2.kStep = static_cast<uint16_t>(ShiftCeilDiv16(t->mStep));
    // mStep = CeilDiv(baseUseK_, k0) in shifted units (= 1 for HO*WO=16)
    load2dParamsV2.mStep = static_cast<uint16_t>(ShiftCeilDiv16(currentKL0));
    // srcStride = alignedL1UseKa / k0 in shifted units (= 1 for alignedL1UseKa=16)
    load2dParamsV2.srcStride = static_cast<int32_t>(ShiftDiv16(ctx.alignedL1UseKa_));
    // dstStride = kStep in shifted units (= 1)
    load2dParamsV2.dstStride = static_cast<uint16_t>(ShiftCeilDiv16(t->mStep));
    load2dParamsV2.ifTranspose = true;

    // Convert LoadData2DParamsV2 to Load2DBitModeParam for bit-mode LoadData call
    Load2DBitModeParam param(load2dParamsV2);

    // For first iteration, config1 (srcStride/dstStride) is already set by the constructor.
    // For subsequent iterations, only config0 (positions/steps) may change,
    // but for our single-tile demo, isFirst is always true.
    if (!isFirst) {
        // Update mStep for tail K iterations
        load2dParamsV2.mStep = static_cast<uint16_t>(ShiftCeilDiv16(currentKL0));
        param.SetMStep(load2dParamsV2.mStep);
    }

    LoadData<TPosition::A2, TPosition::A1, typename Ctx::SrcT>(ctx.al0, ctx.al1, param);
}

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
// ============================================================================
// LoadBL0: fmap from L1 to L0B via Load3D (B matrix im2col)
// ============================================================================
// This is THE key computation: B-side Load3D performs im2col on the feature
// map data, using actual convolution parameters (stride, filter, dilation).
//
// Key difference from the old approach:
//   Old:  fMatrixCtrl=1 (RIGHT register), Fmatrix set via Load3DSetFMatrixBCal
//   New:  fMatrixCtrl=0 (LEFT register), Fmatrix set via Load3DSetFMatrixCal
//
// On dav-3510, we use SetLoadDataRepeat with LoadDataRepeatParam (which includes
// dstStride on this architecture). The dstStride for B-side Load3D is based on
// the N dimension: dstStride = CeilDiv(nL0, 16) = CeilDiv(144, 16) = 9.
//
// B-side Load3D parameters from production code (InitLoadToB2Params + CalcParamsL12L0b):
//   kExtension = baseUseN_ = nL0 = 144 (N = CI*KH*KW)
//   mExtension = baseUseK_ = kL0 = 16  (K = HO*WO)
//   kStartPt = 0 (first N iteration)
//   mStartPt = 0 (first K iteration)
//   fMatrixCtrl = 0 (LEFT register)
//   channelSize = Ceil(kStartPt + baseUseN_, hkwk16) * 16 = 16
//     where hkwk16 = hwK_ * n0 = KH*KW * 16 = 144
template <class Ctx>
__aicore__ inline void LoadBL0(Ctx &ctx, bool isFirst)
{
    const auto *t = ctx.convTiling;
    uint64_t currentKL0 = (ctx.kIter == ctx.maxKL0Iter) ? ctx.kL0Tail : t->kL0;
    uint64_t posK = ctx.kIter * t->kL0;

    // channelSize calculation from production CalcParamsL12L0b:
    // hkwk16 = hwK_ * n0 = KH*KW * 16 (for fp16, n0 = k0 = 16)
    // channelSize = Ceil(kStartPt + baseUseN_, hkwk16) * channelSize_const
    uint64_t hwK = t->kernelH * t->kernelW;  // = KH*KW = 9 (for dhwK=1)
    uint64_t hkwk16 = hwK * Ctx::k0;           // = 9 * 16 = 144
    uint64_t kStartPt = 0;                      // first N iteration
    uint64_t channelSize = CeilDiv(kStartPt + t->nL0, hkwk16) * t->channelSize;
    // = CeilDiv(0 + 144, 144) * 16 = 1 * 16 = 16

    uint16_t mExtension = static_cast<uint16_t>(currentKL0);
    uint16_t mStartPt = static_cast<uint16_t>(posK);

    Load3DBitModeParam param;
    if (isFirst) {
        uint64_t xmtmp = ((mExtension & MASK_16) << MSTEP_OFFSET) |
                         ((uint64_t)(mStartPt & MASK_16) << POSM_OFFSET);
        // config1: actual conv stride, filter, dilation; fMatrixCtrl=0 (LEFT register)
        uint64_t xt = ((uint64_t(t->strideW) & MASK_6) << STRIDEW_OFFSET) |
                      ((uint64_t(t->strideH) & MASK_6) << STRIDEH_OFFSET) |
                      ((uint64_t(t->kernelW) & MASK_8) << FILTERW_OFFSET) |
                      ((uint64_t(t->kernelW) & NINTH_BIT_MASK) << FILTERSIZEW_OFFSET) |
                      ((uint64_t(t->kernelH) & MASK_8) << FILTERH_OFFSET) |
                      ((uint64_t(t->kernelH) & NINTH_BIT_MASK) << FILTERSIZEH_OFFSET) |
                      ((uint64_t(t->dilationW) & MASK_8) << DILATIONW_OFFSET) |
                      ((uint64_t(t->dilationH) & MASK_8) << DILATIONH_OFFSET) |
                      ((uint64_t(0) & MASK_1) << TRANSPOSE_OFFSET) |
                      ((uint64_t(0) & MASK_1) << FMATRIXCTRL_OFFSET) |  // LEFT register
                      ((uint64_t(channelSize) & MASK_16) << CHANNELSIZE_OFFSET);
        param.SetConfig1(xt);

        // kExtension = nL0 (N dimension for L0B, = CI*KH*KW aligned)
        uint16_t kExtension = static_cast<uint16_t>(t->nL0);
        uint64_t xm = ((kExtension & MASK_16) << 0) | ((uint64_t)(0) << POSK_OFFSET) | xmtmp;
        param.SetConfig0(xm);

        // On dav-3510, LoadDataRepeatParam includes dstStride field
        // dstStride = CeilDiv(nL0, BLOCK_L0_N) = CeilDiv(144, 16) = 9
        uint16_t dstStride = static_cast<uint16_t>(CeilDiv(t->nL0, BLOCK_L0_N));
        LoadDataRepeatParam repeatParams = {0, 1, 0, dstStride};
        SetLoadDataRepeat(repeatParams);
    } else {
        uint64_t xmtmp = ((mExtension & MASK_16) << MSTEP_OFFSET) |
                         ((uint64_t)(mStartPt & MASK_16) << POSM_OFFSET);
        uint64_t xm = ((0 & MASK_16) << 0) | ((uint64_t)(0) << POSK_OFFSET) | xmtmp;
        param.SetConfig0(xm);
    }
    LoadData<TPosition::B2, TPosition::B1, typename Ctx::SrcT>(ctx.bl0, ctx.bl1, param);
}

### 3.7 矩阵乘累加：Mad 与 CopyOut

**Mad**：调用 `Mmad` 指令执行矩阵乘累加。

M/K/N 维度映射（与前向卷积相反）：
- **M = CO = 16**（梯度输出的通道数）
- **K = HO × WO = 16**（空间位置数）
- **N = CI × KH × KW = 144**（输入通道 × 核空间尺寸）

关键参数：
- `cmatrixInitVal = (kIter == 0)`：第一次 K 迭代时清零 L0C。
- `unitFlag`：最后一次 K 迭代设置 `UNIT_FLAG_ENABLE_WITH_FLIP`。

**CopyOut**：调用 `Fixpipe` 将 L0C 的 fp32 结果转换为 fp16 并写回 GM。

与前向卷积的关键差异：
- **dstStride = orgCi（= CI = 16）**，而非前向的 `HO × WO`。因为权重梯度的输出布局是 `(CO, CI, KH, KW)`，步长为 CI。
- **布局为 COLUMN_MAJOR**，与权重梯度的 NCDHW 格式对应。
- **quantPre = F322F16**：fp32→fp16 量化。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
template <class Ctx>
__aicore__ inline void Mad(Ctx &ctx)
{
    MmadParams p;
    // M = CO, K = HO*WO, N = CI*KH*KW
    p.m = ctx.currentML0Align;
    p.n = ctx.currentNL0Align;
    p.k = (ctx.kIter == ctx.maxKL0Iter) ? ctx.kL0Tail : ctx.convTiling->kL0;
    p.cmatrixInitVal = (ctx.kIter == 0);
    p.cmatrixSource = false;
    p.unitFlag = (ctx.kIter == ctx.maxKL0Iter) ? UNIT_FLAG_ENABLE_WITH_FLIP : UNIT_FLAG_ENABLE_ONLY;
    Mmad<typename Ctx::L0cT, typename Ctx::SrcT, typename Ctx::SrcT>(
        ctx.cl0, ctx.al0, ctx.bl0, p);
}

template <class Ctx>
__aicore__ inline void CopyOut(Ctx &ctx, const GlobalTensor<typename Ctx::DstT> &output)
{
    // Production code pattern for dkhkwk=1 (LoadL0c2GmDkhkwkEqOne):
    // COLUMN_MAJOR with dnNum=1, srcNzMatrixStride=0, dstDnMatrixStride=0
    // dstStride = cinG_ = CI = 16 (stride between CO groups in NCDHW output)
    FixpipeParamsC310<CO2Layout::COLUMN_MAJOR> p;
    p.mSize = ctx.currentML0;
    p.params.srcNzMatrixStride = 0;
    p.params.dnNum = 1;
    p.params.dstDnMatrixStride = 0;
    p.nSize = ctx.currentNL0Align;
    p.srcStride = AlignB(p.mSize, BLOCK_L0_M);
    p.dstStride = ctx.convTiling->orgCi;  // cinG_ = CI = 16
    p.params.srcNzC0Stride = 1;
    p.quantPre = QuantMode_t::F322F16;
    p.deqScalar = DEQ_SCALAR_ONE;
    p.unitFlag = UNIT_FLAG_ENABLE_WITH_FLIP;

    Fixpipe<typename Ctx::DstT, typename Ctx::L0cT, CFG_COLUMN_MAJOR>(
        output[0], ctx.cl0, p);
}

### 3.8 迭代控制：IterateOne 与 EndConv

**IterateOne**：执行一次完整的权重梯度 tile 计算：

1. 调用 `LoadAL1` / `LoadBL1` 将 gradOutput 和 fmap 从 GM 搬运到 L1。
2. 进入 K 维度循环：每次迭代调用 `LoadAL0`（Load2D transpose）、`LoadBL0`（Load3D im2col）、`Mad`。
3. 使用 `WaitFlag` / `SetFlag` 在 MTE1 和 M 单元之间同步。
4. 调用 `CopyOut` 将 L0C 结果写回 GM。

**EndConv**：释放 L1 队列中的所有事件资源。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
template <class Ctx>
__aicore__ inline void IterateOne(Ctx &ctx, const GlobalTensor<typename Ctx::DstT> &output)
{
    ctx.cl0 = ctx.wholeCl0Tensor;
    ctx.kIter = 0;

    LoadAL1(ctx);
    LoadBL1(ctx);
    ctx.al1 = ctx.queueAL1.template DeQue<typename Ctx::SrcT>();
    ctx.bl1 = ctx.queueBL1.template DeQue<typename Ctx::SrcT>();

    event_t ev = static_cast<event_t>(L0_SYBC_DB_CLOSE);
    bool isFirst = true;
    while (ctx.kIter < ctx.ddr2l0LoopK) {
        AscendC::WaitFlag<AscendC::HardEvent::M_MTE1>(ev);
        ctx.al0 = ctx.wholeAl0Tensor;
        ctx.kAL0Iter = ctx.kIter;
        LoadAL0(ctx, isFirst);
        ctx.bl0 = ctx.wholeBl0Tensor;
        LoadBL0(ctx, isFirst);
        AscendC::SetFlag<AscendC::HardEvent::MTE1_M>(ev);
        AscendC::WaitFlag<AscendC::HardEvent::MTE1_M>(ev);
        Mad(ctx);
        AscendC::SetFlag<AscendC::HardEvent::M_MTE1>(ev);
        ctx.kIter++;
        isFirst = false;
    }

    ctx.queueAL1.FreeTensor(ctx.al1);
    ctx.queueBL1.FreeTensor(ctx.bl1);

    if ASCEND_IS_AIC {
        CopyOut(ctx, output);
    }
}

template <class Ctx>
__aicore__ inline void EndConv(Ctx &ctx)
{
    if ASCEND_IS_AIC {
        ctx.queueAL1.FreeAllEvent();
        ctx.queueBL1.FreeAllEvent();
    }
}

### 3.9 ConvBpFilterBase 类与核函数入口

**关闭 namespace conv_bp_filter**，定义三个 `constexpr` 变量指定梯度输出、fmap 和权重梯度的数据布局（均为 NCHW）。

**ConvBpFilterBase**：模板类，封装了 `RunConvBpFilterKernel` 方法：
1. 设置 GM 指针（doutGm、fmapGm、dwGm）。
2. 调用 `InitContext` 初始化上下文。
3. 调用 `IterateOne` 执行权重梯度计算。
4. 调用 `EndConv` 释放资源。

**conv3d_backprop_filter_v2_kernel**：全局核函数入口，使用 `GET_TILING_DATA` 宏从 GM 读取 Tiling 参数，实例化 `ConvBpFilterBase` 并调用 `RunConvBpFilterKernel`。核函数签名为 `(dout, x, dw, workspace, tilingGm)`，不含 `bias` 和 `offset_w` 参数。

`#endif` 结束设备侧编译块。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
}  // namespace conv_bp_filter

using namespace conv_bp_filter;

constexpr ConvFormat fmapFormat = ConvFormat::NCHW;
constexpr ConvFormat doutFormat = ConvFormat::NCHW;
constexpr ConvFormat dwFormat = ConvFormat::NCHW;

template <class SRC_TYPE, class DST_TYPE>
class ConvBpFilterBase {
public:
    using SRC_T = typename SRC_TYPE::T;
    using DST_T = typename DST_TYPE::T;

    __aicore__ inline void RunConvBpFilterKernel(GM_ADDR dout, GM_ADDR x, GM_ADDR dw,
                                                    const ConvBpFilterTilingData &tilingData)
    {
        doutGm.SetGlobalBuffer(reinterpret_cast<__gm__ SRC_T *>(dout));
        fmapGm.SetGlobalBuffer(reinterpret_cast<__gm__ SRC_T *>(x));
        dwGm.SetGlobalBuffer(reinterpret_cast<__gm__ DST_T *>(dw));

        InitContext(ctx, &tilingData);

        ctx.doutGm = doutGm;
        ctx.fmapGm = fmapGm;

        if ASCEND_IS_AIC {
            IterateOne(ctx, dwGm);
        }
        EndConv(ctx);
    }

private:
    conv_bp_filter::ConvBpFilterContext<SRC_TYPE, DST_TYPE> ctx;
    GlobalTensor<SRC_T> doutGm;
    GlobalTensor<SRC_T> fmapGm;
    GlobalTensor<DST_T> dwGm;
};
#endif

---
## 4. Host侧调用代码

Host侧代码负责：填充 Tiling 结构体、验证片上空间、分配设备内存、启动核函数、回收结果。

### 4.1 卷积形状常量与辅助函数

定义本 Demo 使用的卷积形状参数。与前向 Conv2D Demo 的关键差异：

- **HI = 6, WI = 6**（而非前向的 HI=8, WI=8），使得 **K = HO×WO = 4×4 = 16**，恰好是 C0（fp16=16）的倍数，满足 A 侧 Load2D(transpose) 和 B 侧 Load3D 的对齐要求。

`Fp16ToFloat` 是一个纯软件的 fp16 解码函数，用于在主机侧将输出结果转换为 float 以便打印。支持 subnormal（次正规）值、无穷大和 NaN 的正确转换。

新增常量 `FP16_ONE = 0x3C00`（IEEE-754 half-precision 1.0 的位模式），用于初始化输入数据，替代原来的魔术数字 `0x3C00`。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
// ============================================================================
// Shape constants
// ============================================================================
// Using shapes where K = HO*WO is a multiple of C0 (16 for half),
// since A-side Load2D with transpose requires K-aligned transfers.
//   Forward Conv2D:  M=HO*WO=36, K=CI*KH*KW=144, N=CO=16
//   Backprop Filter:    M=CO=16,    K=HO*WO=16,      N=CI*KH*KW=144
constexpr uint32_t BATCH = 1;
constexpr uint32_t CI = 16;
constexpr uint32_t HI = 6;
constexpr uint32_t WI = 6;
constexpr uint32_t CO = 16;
constexpr uint32_t KH = 3;
constexpr uint32_t KW = 3;
constexpr uint32_t STRIDE_H = 1;
constexpr uint32_t STRIDE_W = 1;
constexpr uint32_t DILATION_H = 1;
constexpr uint32_t DILATION_W = 1;
constexpr uint32_t PAD_TOP = 0;
constexpr uint32_t PAD_LEFT = 0;
constexpr uint32_t PAD_RIGHT = 0;
constexpr uint32_t PAD_DOWN = 0;
constexpr uint32_t HO = (HI + PAD_TOP + PAD_DOWN - DILATION_H * (KH - 1) - 1) / STRIDE_H + 1;
constexpr uint32_t WO = (WI + PAD_LEFT + PAD_RIGHT - DILATION_W * (KW - 1) - 1) / STRIDE_W + 1;

__global__ __aicore__ void conv3d_backprop_filter_v2_kernel(GM_ADDR dout, GM_ADDR x,
    GM_ADDR dw, __kfc_workspace__ GM_ADDR workspace, GM_ADDR tilingGm)
{
#ifdef __CCE_AICORE__
    GET_TILING_DATA(tilingData, tilingGm);

    using srcType = ConvType<TPosition::GM, fmapFormat, half>;
    using dstType = ConvType<TPosition::GM, dwFormat, half>;

    ConvBpFilterBase<srcType, dstType> baseConvBpFilter;
    baseConvBpFilter.RunConvBpFilterKernel(dout, x, dw, tilingData);
#endif
}

constexpr uint8_t BLOCK_SIZE = 16;
// IEEE-754 half-precision bit pattern for 1.0
constexpr uint16_t FP16_ONE = 0x3C00;

float Fp16ToFloat(uint16_t h)
{
    uint32_t sign = (h >> 15) & 0x1;
    uint32_t exp = (h >> 10) & 0x1F;
    uint32_t frac = h & 0x3FF;
    float result;
    if (exp == 0) {
        if (frac == 0) {
            result = 0.0f;
        } else {
            // Subnormal: no implicit leading 1; value = frac * 2^(-14) / 1024 = frac * 2^(-24)
            // After normalization, the implicit leading 1 emerges when shifting to normalized range
            result = std::ldexp(static_cast<float>(frac), -24);
        }
    } else if (exp == 31) {
        result = (frac == 0) ? std::numeric_limits<float>::infinity() : std::numeric_limits<float>::quiet_NaN();
    } else {
        // Normalized: implicit leading 1; value = (1 + frac/1024) * 2^(exp-15)
        result = std::ldexp(1.0f + static_cast<float>(frac) / 1024.0f, static_cast<int>(exp) - 15);
    }
    return sign ? -result : result;
}

### 4.2 SetTilingData

`SetTilingData` 根据卷积形状常量填充 `ConvBpFilterTilingData` 结构体。

关键字段说明（与前向 Conv2D 的差异）：

- **singleCoreM = CO**：反向传播中 M = 输出通道数（前向中 M = HO×WO）。
- **singleCoreK = HO × WO**：反向传播中 K = 空间位置数（前向中 K = CI×KH×KW）。
- **singleCoreN = CI × KH × KW**：反向传播中 N = 输入通道×核空间（前向中 N = CO）。
- **kAL1 = HO × WO**：A 侧 L1 的 K 维度 = 空间位置数（前向中 = CI×KH×KW）。
- **kBL1 = HO × WO**：B 侧 L1 的 K 维度（与 kAL1 相等）。
- **nBL1 = CI × KH × KW**：B 侧 L1 的 N 维度。
- **mStep = AlignUp(CO, 16)**：M 对齐到 16（前向中 mStep = AlignUp(HO×WO, 16)）。
- **nL0 = AlignUp(CI×KH×KW, 16)**：N 对齐到 16。
- **kStep**：Load2D kStep（shifted 单位），= CeilDiv(CO, 16)。
- **alignedL1UseKa**：A 侧 L1 NZ 格式对齐 K = CeilDiv(HO×WO, 16) × 16。
- **aL1SpaceSize**：gradOutput NZ 格式在 L1 中的空间大小。
- **dstStride（Fixpipe）= orgCi = CI**：权重梯度输出步长（前向中 = HO×WO）。

代码中标注了 `[QUIZ]` 的位置是本章课后练习的填空点。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
void SetTilingData(ConvBpFilterTilingData* tiling)
{
    tiling->padTop = PAD_TOP;
    tiling->padLeft = PAD_LEFT;
    tiling->padRight = PAD_RIGHT;
    tiling->padDown = PAD_DOWN;

    tiling->orgHi = HI;
    tiling->orgWi = WI;
    tiling->orgHo = HO;
    tiling->orgWo = WO;
    tiling->orgCi = CI;
    tiling->orgCo = CO;
    tiling->singleCoreBatch = BATCH;

    // Backprop filter M/K/N mapping:
    //   M = CO = 16       (output channels from gradOutput)
    //   K = HO * WO = 16  (spatial positions for reduction)
    //   N = CI * KH * KW = 144 (from fmap im2col)
    // [QUIZ 1/4] singleCoreM: M dimension for backprop filter.
    //   In forward Conv2D, M = HO*WO. In backprop filter, M = CO.
    //   Hint: use CO
    /* BEGIN_TODO */
    tiling->singleCoreM = CO;
    /* END_TODO */
    // [QUIZ 2/4] singleCoreK: K dimension for backprop filter.
    //   In forward, K = CI*KH*KW. In backprop, K = HO*WO.
    //   Hint: use HO and WO
    /* BEGIN_TODO */
    tiling->singleCoreK = HO * WO;
    /* END_TODO */
    tiling->singleCoreN = CI * KH * KW;

    // kAL1: K dimension loaded into L1 for A-side (gradOutput)
    tiling->kAL1 = HO * WO;
    // kBL1: K dimension loaded into L1 for B-side (fmap)
    tiling->kBL1 = HO * WO;
    tiling->nBL1 = CI * KH * KW;

    // M aligned to next multiple of 16 for Mmad
    tiling->mStep = ((CO + 15) / 16) * 16;
    // K for L0 buffers (= HO*WO, already multiple of 16)
    tiling->kL0 = HO * WO;
    // N aligned to next multiple of 16 for Mmad
    // CI*KH*KW = 16*3*3 = 144, which is already 16-aligned (9*16)
    // [QUIZ 3/4] nL0: N dimension aligned to next multiple of 16.
    //   N = CI*KH*KW = 16*3*3 = 144, already 16-aligned (9*16).
    //   Hint: AlignUp(CI * KH * KW, 16)
    /* BEGIN_TODO */
    tiling->nL0 = ((CI * KH * KW + 15) / 16) * 16;
    /* END_TODO */

    // kStep: Load2D kStep parameter (shifted units)
    // = CeilDiv(baseUseM, m0) = CeilDiv(CO, 16) = 1
    // In actual elements: 1 * 16 = 16
    tiling->kStep = (CO + BLOCK_SIZE - 1) / BLOCK_SIZE;

    // alignedL1UseKa: Aligned K for A-side L1 in NZ format
    // = CeilDiv(kAL1, k0) * k0 = CeilDiv(16, 16) * 16 = 16
    tiling->alignedL1UseKa = ((HO * WO + BLOCK_SIZE - 1) / BLOCK_SIZE) * BLOCK_SIZE;

    tiling->orgHixWi = HI * WI;
    tiling->kernelHxkernelW = KH * KW;

    // AL1 space: gradOutput in NZ format in L1
    // Size = dstNzC0Stride * dValue * sizeof(half)
    // dstNzC0Stride = CeilDiv(nValue, k0) * k0 = CeilDiv(16, 16) * 16 = 16
    // dValue = CO = 16
    // So: 16 * 16 * 2 = 512 bytes
    // [QUIZ 4/4] aL1SpaceSize: gradOutput NZ format L1 space size.
    //   Size = dstNzC0Stride * dValue * sizeof(half)
    //   dstNzC0Stride = CeilDiv(nValue, k0) * k0 = CeilDiv(HO*WO, 16) * 16
    //   dValue = CO = 16
    //   Hint: CeilDiv(HO*WO, 16) * 16 * CO * sizeof(uint16_t)
    /* BEGIN_TODO */
    tiling->aL1SpaceSize = ((HO * WO + BLOCK_SIZE - 1) / BLOCK_SIZE) * BLOCK_SIZE * CO * sizeof(uint16_t);
    /* END_TODO */

    // BL1 space: fmap patch (hiLoad * wiLoad * CI elements)
    uint32_t hiLoad = (HO - 1) * STRIDE_H + (1 + (KH - 1) * DILATION_H);
    uint32_t wiLoad = (WO - 1) * STRIDE_W + (1 + (KW - 1) * DILATION_W);
    tiling->bL1SpaceSize = hiLoad * wiLoad * CI;

    tiling->cinAInCore = tiling->orgCo;
    tiling->cinBInCore = tiling->orgCi;

    tiling->channelSize = 16;

    tiling->kernelH = KH;
    tiling->kernelW = KW;
    tiling->strideH = STRIDE_H;
    tiling->strideW = STRIDE_W;
    tiling->dilationH = DILATION_H;
    tiling->dilationW = DILATION_W;
    tiling->offsetx = 0;
}

static bool CheckTilingSpace(const ConvBpFilterTilingData* t)
{
    const uint32_t fp16 = sizeof(uint16_t);
    const uint32_t fp32 = sizeof(float);

    uint64_t al1Bytes = static_cast<uint64_t>(t->aL1SpaceSize);
    uint64_t bl1Bytes = static_cast<uint64_t>(t->bL1SpaceSize) * fp16;

    const uint64_t L1_SIZE = 512 * 1024ULL;
    uint64_t l1Total = al1Bytes + bl1Bytes;

    uint32_t mAlign = t->mStep;
    uint32_t kAlign = t->kL0;
    uint32_t nAlign = t->nL0;

    uint64_t al0Bytes = (uint64_t)mAlign * kAlign * fp16;
    uint64_t bl0Bytes = (uint64_t)kAlign * nAlign * fp16;
    uint64_t cl0Bytes = (uint64_t)mAlign * nAlign * fp32;

    const uint64_t L0A_LIMIT = 64 * 1024ULL;
    const uint64_t L0B_LIMIT = 64 * 1024ULL;
    const uint64_t L0C_LIMIT = 256 * 1024ULL;

    bool ok = true;
    if (l1Total > L1_SIZE) {
        std::cerr << "[TILING ERROR] L1 overflow: need " << l1Total
                  << " bytes (AL1=" << al1Bytes << " + BL1=" << bl1Bytes
                  << "), limit=" << L1_SIZE << "\n";
        ok = false;
    }
    if (al0Bytes > L0A_LIMIT) {
        std::cerr << "[TILING ERROR] L0A overflow: need " << al0Bytes
                  << " bytes (M=" << mAlign << " K=" << kAlign
                  << "), limit=" << L0A_LIMIT << "\n";
        ok = false;
    }
    if (bl0Bytes > L0B_LIMIT) {
        std::cerr << "[TILING ERROR] L0B overflow: need " << bl0Bytes
                  << " bytes (K=" << kAlign << " N=" << nAlign
                  << "), limit=" << L0B_LIMIT << "\n";
        ok = false;
    }
    if (cl0Bytes > L0C_LIMIT) {
        std::cerr << "[TILING ERROR] L0C overflow: need " << cl0Bytes
                  << " bytes (M=" << mAlign << " N=" << nAlign
                  << "), limit=" << L0C_LIMIT << "\n";
        ok = false;
    }
    if (ok) {
        std::cout << "[TILING OK] L1=" << l1Total << "/" << L1_SIZE
                  << "  L0A=" << al0Bytes << "/" << L0A_LIMIT
                  << "  L0B=" << bl0Bytes << "/" << L0B_LIMIT
                  << "  L0C=" << cl0Bytes << "/" << L0C_LIMIT << "\n";
        std::cout << "[TILING INFO] Mmad dims: M=" << mAlign
                  << " K=" << kAlign << " N=" << nAlign
                  << " (M=CO, K=HO*WO, N=CI*KH*KW)\n";
        std::cout << "[TILING INFO] A=Load2D(transpose,gradOutput->L0A), B=Load3D(im2col,fmap->L0B,LEFT Fmatrix)\n";
    }
    return ok;
}

void Dump4DTensor(std::ofstream& ofs, const uint16_t* data,
    uint32_t d0, uint32_t d1, uint32_t d2, uint32_t d3,
    const char* dim0, const char* dim1)
{
    for (uint32_t i = 0; i < d0; ++i) {
        for (uint32_t j = 0; j < d1; ++j) {
            ofs << dim0 << "=" << i << ", " << dim1 << "=" << j << ":" << std::endl;
            for (uint32_t k = 0; k < d2; ++k) {
                for (uint32_t l = 0; l < d3; ++l) {
                    uint32_t idx = ((i * d1 + j) * d2 + k) * d3 + l;
                    ofs << Fp16ToFloat(data[idx]) << " ";
                }
                ofs << std::endl;
            }
        }
    }
}

void WriteInputMatrix(const uint16_t* fmapData, const uint16_t* doutData)
{
    std::ofstream ofs("input_matrix.txt");
    ofs << "Fmap (Input X) shape: [" << BATCH << ", " << CI << ", " << HI << ", " << WI << "]" << std::endl;
    Dump4DTensor(ofs, fmapData, BATCH, CI, HI, WI, "N", "C");
    ofs << "\nGradOutput (dL/dY) shape: [" << BATCH << ", " << CO << ", " << HO << ", " << WO << "]" << std::endl;
    Dump4DTensor(ofs, doutData, BATCH, CO, HO, WO, "N", "CO");
    ofs.close();
    std::cout << "Input matrices written to input_matrix.txt" << std::endl;
}

void WriteOutputMatrix(const uint16_t* dwData)
{
    std::ofstream ofs("output_matrix.txt");
    ofs << "GradWeight (dL/dW) shape: [" << CO << ", " << CI << ", " << KH << ", " << KW << "]" << std::endl;
    Dump4DTensor(ofs, dwData, CO, CI, KH, KW, "CO", "CI");
    ofs.close();
    std::cout << "Output matrix written to output_matrix.txt" << std::endl;
}

struct HostDeviceBuffer {
    uint8_t* host = nullptr;
    uint8_t* device = nullptr;
};

aclError CheckAclError(aclError ret, const char* msg)
{
    if (ret != ACL_SUCCESS) {
        std::cerr << "[ACL ERROR] " << msg << " failed: ret=" << ret << std::endl;
    }
    return ret;
}

aclError InitAclRuntime(int32_t deviceId, aclrtStream& stream)
{
    aclError ret = aclInit(nullptr);
    if (CheckAclError(ret, "aclInit") != ACL_SUCCESS) return ret;
    ret = aclrtSetDevice(deviceId);
    if (CheckAclError(ret, "aclrtSetDevice") != ACL_SUCCESS) return ret;
    ret = aclrtCreateStream(&stream);
    if (CheckAclError(ret, "aclrtCreateStream") != ACL_SUCCESS) return ret;
    return ACL_SUCCESS;
}

HostDeviceBuffer AllocAndInitInputBuffer(size_t size, uint16_t fillValue)
{
    HostDeviceBuffer buf;
    if (aclrtMallocHost((void**)&buf.host, size) != ACL_SUCCESS || buf.host == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMallocHost failed for input buffer (" << size << " bytes)\n";
        return {};
    }
    if (aclrtMalloc((void**)&buf.device, size, ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS || buf.device == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMalloc failed for input device buffer (" << size << " bytes)\n";
        aclrtFreeHost(buf.host);
        return {};
    }
    uint16_t* halfData = reinterpret_cast<uint16_t*>(buf.host);
    for (size_t i = 0; i < size / sizeof(uint16_t); ++i) {
        halfData[i] = fillValue;
    }
    aclrtMemcpy(buf.device, size, buf.host, size, ACL_MEMCPY_HOST_TO_DEVICE);
    return buf;
}

HostDeviceBuffer AllocOutputBuffer(size_t size)
{
    HostDeviceBuffer buf;
    if (aclrtMallocHost((void**)&buf.host, size) != ACL_SUCCESS || buf.host == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMallocHost failed for output buffer (" << size << " bytes)\n";
        return {};
    }
    if (aclrtMalloc((void**)&buf.device, size, ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS || buf.device == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMalloc failed for output device buffer (" << size << " bytes)\n";
        aclrtFreeHost(buf.host);
        return {};
    }
    return buf;
}

uint8_t* AllocWorkspaceBuffer(size_t size)
{
    uint8_t* ptr = nullptr;
    if (aclrtMalloc((void**)&ptr, size, ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS || ptr == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMalloc failed for workspace (" << size << " bytes)\n";
        return nullptr;
    }
    return ptr;
}

HostDeviceBuffer AllocAndCopyTilingBuffer(const ConvBpFilterTilingData& tilingBuf, size_t size)
{
    HostDeviceBuffer buf;
    if (aclrtMallocHost((void**)&buf.host, size) != ACL_SUCCESS || buf.host == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMallocHost failed for tiling buffer (" << size << " bytes)\n";
        return {};
    }
    if (aclrtMalloc((void**)&buf.device, size, ACL_MEM_MALLOC_HUGE_FIRST) != ACL_SUCCESS || buf.device == nullptr) {
        std::cerr << "[ACL ERROR] aclrtMalloc failed for tiling device buffer (" << size << " bytes)\n";
        aclrtFreeHost(buf.host);
        return {};
    }
    memcpy(buf.host, &tilingBuf, size);
    aclrtMemcpy(buf.device, size, buf.host, size, ACL_MEMCPY_HOST_TO_DEVICE);
    return buf;
}

void FreeHostDeviceBuffer(const HostDeviceBuffer& buf)
{
    if (buf.device != nullptr) aclrtFree(buf.device);
    if (buf.host != nullptr) aclrtFreeHost(buf.host);
}

void CleanupAclResources(const HostDeviceBuffer& fmap, const HostDeviceBuffer& dout,
    const HostDeviceBuffer& dw, uint8_t* workspace, const HostDeviceBuffer& tiling,
    aclrtStream stream, int32_t deviceId)
{
    FreeHostDeviceBuffer(fmap);
    FreeHostDeviceBuffer(dout);
    FreeHostDeviceBuffer(dw);
    if (workspace != nullptr) aclrtFree(workspace);
    FreeHostDeviceBuffer(tiling);
    if (stream != nullptr) aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
}

int32_t main(int32_t argc, char* argv[])
{
    size_t fmapSize = BATCH * CI * HI * WI * sizeof(uint16_t);
    size_t doutSize = BATCH * CO * HO * WO * sizeof(uint16_t);
    size_t dwSize = CO * CI * KH * KW * sizeof(uint16_t);
    size_t tilingSize = sizeof(ConvBpFilterTilingData);
    size_t workspaceSize = 16 * 1024 * 1024;
    uint32_t numBlocks = 1;

    ConvBpFilterTilingData tilingBuf;
    SetTilingData(&tilingBuf);

    if (!CheckTilingSpace(&tilingBuf)) {
        std::cerr << "Tiling validation failed. Adjust shape constants or tiling params.\n";
        return 1;
    }

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    if (InitAclRuntime(deviceId, stream) != ACL_SUCCESS) {
        std::cerr << "ACL runtime initialization failed.\n";
        return 1;
    }

    HostDeviceBuffer fmap = AllocAndInitInputBuffer(fmapSize, FP16_ONE);
    HostDeviceBuffer dout = AllocAndInitInputBuffer(doutSize, FP16_ONE);
    HostDeviceBuffer dw = AllocOutputBuffer(dwSize);
    uint8_t* workspace = AllocWorkspaceBuffer(workspaceSize);
    HostDeviceBuffer tiling = AllocAndCopyTilingBuffer(tilingBuf, tilingSize);

    if (fmap.host == nullptr || dout.host == nullptr || dw.host == nullptr ||
        workspace == nullptr || tiling.host == nullptr) {
        std::cerr << "Memory allocation failed.\n";
        CleanupAclResources(fmap, dout, dw, workspace, tiling, stream, deviceId);
        return 1;
    }

    conv3d_backprop_filter_v2_kernel<<<numBlocks, nullptr, stream>>>(
        dout.device, fmap.device, dw.device, workspace, tiling.device);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(dw.host, dwSize, dw.device, dwSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::cout << "Conv3D Backprop Filter v2 kernel execution completed." << std::endl;

    WriteInputMatrix(reinterpret_cast<uint16_t*>(fmap.host), reinterpret_cast<uint16_t*>(dout.host));
    WriteOutputMatrix(reinterpret_cast<uint16_t*>(dw.host));

    CleanupAclResources(fmap, dout, dw, workspace, tiling, stream, deviceId);
    return 0;
}

### 4.3 CheckTilingSpace

在启动核函数之前，必须验证 Tiling 参数是否适合 NPU 的片上 SRAM。

与前向 Conv2D 的关键差异：M/K/N 维度映射不同，导致各缓冲区的计算公式不同。优化后的 `CheckTilingSpace` 直接从 tiling 结构体读取 `mStep`、`kL0`、`nL0`、`aL1SpaceSize`、`bL1SpaceSize` 等字段，不再重新计算这些值：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>缓冲区</strong></td>
  <td align="center"><strong>前向 Conv2D</strong></td>
  <td align="center"><strong>Backprop Filter</strong></td>
</tr>
<tr>
  <td align="left">L0A</td>
  <td align="left">AlignUp(HO×WO, 16) × kL0 × fp16</td>
  <td align="left"><strong>t->mStep × t->kL0 × fp16</strong></td>
</tr>
<tr>
  <td align="left">L0B</td>
  <td align="left">AlignUp(CO, 16) × kL0 × fp16</td>
  <td align="left"><strong>t->kL0 × t->nL0 × fp16</strong></td>
</tr>
<tr>
  <td align="left">L0C</td>
  <td align="left">AlignUp(HO×WO, 16) × AlignUp(CO, 16) × fp32</td>
  <td align="left"><strong>t->mStep × t->nL0 × fp32</strong></td>
</tr>
</table>

代码中标注了 `[QUIZ]` 的位置是本章课后练习的填空点。优化后 quiz 从 5 处减至 3 处，因为 `mAlign/kAlign/nAlign` 和 `hiLoad/wiLoad` 现在直接从 tiling 读取，不再需要手动计算。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
static bool CheckTilingSpace(const ConvBpFilterTilingData* t)
{
    const uint32_t fp16 = sizeof(uint16_t);
    const uint32_t fp32 = sizeof(float);

    // [QUIZ 1/3] AL1 and BL1 space sizes from tiling struct.
    //   al1Bytes = t->aL1SpaceSize (already computed in SetTilingData)
    //   bl1Bytes = t->bL1SpaceSize * fp16
    /* BEGIN_TODO */
    uint64_t al1Bytes = static_cast<uint64_t>(t->aL1SpaceSize);
    uint64_t bl1Bytes = static_cast<uint64_t>(t->bL1SpaceSize) * fp16;
    /* END_TODO */

    const uint64_t L1_SIZE = 512 * 1024ULL;
    uint64_t l1Total = al1Bytes + bl1Bytes;

    // [QUIZ 2/3] Mmad aligned dimensions from tiling struct.
    //   mAlign = t->mStep (M=CO aligned to 16)
    //   kAlign = t->kL0 (K=HO*WO)
    //   nAlign = t->nL0 (N=CI*KH*KW aligned to 16)
    /* BEGIN_TODO */
    uint32_t mAlign = t->mStep;
    uint32_t kAlign = t->kL0;
    uint32_t nAlign = t->nL0;
    /* END_TODO */

    uint64_t al0Bytes = (uint64_t)mAlign * kAlign * fp16;
    uint64_t bl0Bytes = (uint64_t)kAlign * nAlign * fp16;

    // [QUIZ 3/3] CL0 bytes: M_align * N_align * fp32
    //   Note: fp32 = 4 bytes, NOT fp16 = 2 bytes!
    /* BEGIN_TODO */
    uint64_t cl0Bytes = (uint64_t)mAlign * nAlign * fp32;
    /* END_TODO */

    const uint64_t L0A_LIMIT = 64 * 1024ULL;
    const uint64_t L0B_LIMIT = 64 * 1024ULL;
    const uint64_t L0C_LIMIT = 256 * 1024ULL;

    bool ok = true;
    if (l1Total > L1_SIZE) {
        std::cerr << "[TILING ERROR] L1 overflow: need " << l1Total
                  << " bytes (AL1=" << al1Bytes << " + BL1=" << bl1Bytes
                  << "), limit=" << L1_SIZE << "\n";
        ok = false;
    }
    if (al0Bytes > L0A_LIMIT) {
        std::cerr << "[TILING ERROR] L0A overflow: need " << al0Bytes
                  << " bytes (M=" << mAlign << " K=" << kAlign
                  << "), limit=" << L0A_LIMIT << "\n";
        ok = false;
    }
    if (bl0Bytes > L0B_LIMIT) {
        std::cerr << "[TILING ERROR] L0B overflow: need " << bl0Bytes
                  << " bytes (K=" << kAlign << " N=" << nAlign
                  << "), limit=" << L0B_LIMIT << "\n";
        ok = false;
    }
    if (cl0Bytes > L0C_LIMIT) {
        std::cerr << "[TILING ERROR] L0C overflow: need " << cl0Bytes
                  << " bytes (M=" << mAlign << " N=" << nAlign
                  << "), limit=" << L0C_LIMIT << "\n";
        ok = false;
    }
    if (ok) {
        std::cout << "[TILING OK] L1=" << l1Total << "/" << L1_SIZE
                  << "  L0A=" << al0Bytes << "/" << L0A_LIMIT
                  << "  L0B=" << bl0Bytes << "/" << L0B_LIMIT
                  << "  L0C=" << cl0Bytes << "/" << L0C_LIMIT << "\n";
        std::cout << "[TILING INFO] Mmad dims: M=" << mAlign
                  << " K=" << kAlign << " N=" << nAlign
                  << " (M=CO, K=HO*WO, N=CI*KH*KW)\n";
        std::cout << "[TILING INFO] A=Load2D(transpose,gradOutput->L0A), B=Load3D(im2col,fmap->L0B,LEFT Fmatrix)\n";
    }
    return ok;
}

### 4.4 main 函数

`main` 函数完整演示了 AscendC 核函数的调用生命周期：

1. 调用 `SetTilingData` 填充 Tiling 结构体。
2. 调用 `CheckTilingSpace` 验证片上空间，若溢出则提前退出。
3. 初始化 ACL 运行环境（`InitAclRuntime`，带错误检查）。
4. 使用 `HostDeviceBuffer` RAII 结构分配主机和设备内存，用 `FP16_ONE` 常量初始化输入数据（全 1.0 的 fp16）。
5. 分配失败时调用 `CleanupAclResources` 释放已分配资源并退出。
6. 通过 `<<<numBlocks, nullptr, stream>>>` 语法启动核函数（不含 `bias`/`offset_w` 参数）。
7. 将输出（权重梯度 dL/dW）从设备复制回主机，通过 `Dump4DTensor` 辅助函数写入文本文件。
8. 通过 `CleanupAclResources` 释放所有内存和运行时资源（含空指针检查）。

In [ ]:
%%writefile -a Sources/01.04/minimal_demo.asc
int32_t main(int32_t argc, char* argv[])
{
    size_t fmapSize = BATCH * CI * HI * WI * sizeof(uint16_t);
    size_t doutSize = BATCH * CO * HO * WO * sizeof(uint16_t);
    size_t dwSize = CO * CI * KH * KW * sizeof(uint16_t);
    size_t tilingSize = sizeof(ConvBpFilterTilingData);
    size_t workspaceSize = 16 * 1024 * 1024;
    uint32_t numBlocks = 1;

    ConvBpFilterTilingData tilingBuf;
    SetTilingData(&tilingBuf);

    if (!CheckTilingSpace(&tilingBuf)) {
        std::cerr << "Tiling validation failed. Adjust shape constants or tiling params.\n";
        return 1;
    }

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    if (InitAclRuntime(deviceId, stream) != ACL_SUCCESS) {
        std::cerr << "ACL runtime initialization failed.\n";
        return 1;
    }

    HostDeviceBuffer fmap = AllocAndInitInputBuffer(fmapSize, FP16_ONE);
    HostDeviceBuffer dout = AllocAndInitInputBuffer(doutSize, FP16_ONE);
    HostDeviceBuffer dw = AllocOutputBuffer(dwSize);
    uint8_t* workspace = AllocWorkspaceBuffer(workspaceSize);
    HostDeviceBuffer tiling = AllocAndCopyTilingBuffer(tilingBuf, tilingSize);

    if (fmap.host == nullptr || dout.host == nullptr || dw.host == nullptr ||
        workspace == nullptr || tiling.host == nullptr) {
        std::cerr << "Memory allocation failed.\n";
        CleanupAclResources(fmap, dout, dw, workspace, tiling, stream, deviceId);
        return 1;
    }

    conv3d_backprop_filter_v2_kernel<<<numBlocks, nullptr, stream>>>(
        dout.device, fmap.device, dw.device, workspace, tiling.device);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(dw.host, dwSize, dw.device, dwSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::cout << "Conv3D Backprop Filter v2 kernel execution completed." << std::endl;

    WriteInputMatrix(reinterpret_cast<uint16_t*>(fmap.host), reinterpret_cast<uint16_t*>(dout.host));
    WriteOutputMatrix(reinterpret_cast<uint16_t*>(dw.host));

    CleanupAclResources(fmap, dout, dw, workspace, tiling, stream, deviceId);
    return 0;
}

---
## 5. CMakeLists.txt

以下是本项目的 CMake 构建配置。与前向 Conv2D Demo 相比，主要差异：

- **项目名称**：`conv3d_backprop_filter_v2_demo_minimal`（而非 `conv2d_v2_demo_minimal`）。
- **目标架构**：`--npu-arch=dav-3510`（与前向相同）。
- 其他配置（链接库、头文件路径）与前向 Demo 一致。

In [ ]:
%%writefile Sources/01.04/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

find_package(ASC REQUIRED)

project(conv3d_backprop_filter_v2_demo_minimal LANGUAGES ASC CXX)

add_executable(minimal_demo
    minimal_demo.asc
)

target_include_directories(minimal_demo PRIVATE
    ${CMAKE_CURRENT_SOURCE_DIR}
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/basic_api
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/basic_api/reg_compute
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/asc/impl/simt_api
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/include
    ${ASCEND_CANN_PACKAGE_LINUX_PATH}/tikcpp/tikcfw
)

target_link_directories(minimal_demo PRIVATE
    ${ASCEND_CANN_PACKAGE_PATH}/lib64
)

target_link_libraries(minimal_demo PRIVATE
    tiling_api
    register
    platform
    ascendc_runtime
    runtime
    ascendcl
    error_manager
    profapi
    ge_common_base
    unified_dlog
    mmpa
    ascend_dump
    c_sec
    m
    dl
)

target_compile_options(minimal_demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-3510>
)

---
## 6. 构建脚本 build.sh

除了直接在 Jupyter 中运行 CMake 命令，`src_dw/build.sh` 提供了一个等价的一键构建脚本，适合在终端中使用。

```bash
#!/bin/bash
set -e

SCRIPT_DIR=$(cd "$(dirname "$0")" && pwd)
BUILD_DIR=${SCRIPT_DIR}/build
CANN_PATH=${ASCEND_HOME_PATH:-/home/developer/Ascend/cann-9.0.0}

rm -rf ${BUILD_DIR}
mkdir -p ${BUILD_DIR} && cd ${BUILD_DIR}

cmake .. -DASCEND_CANN_PACKAGE_PATH=${CANN_PATH}
make -j$(nproc)

echo "Build success: ${BUILD_DIR}/minimal_demo"
```

**使用方式**：

```bash
# 步骤一：设置环境变量
export ASCEND_HOME_PATH=/your/cann/install/path
bash src_dw/build.sh

# 步骤二：修改脚本中的 CANN_PATH 默认值为你的实际路径
```

> **注意**：脚本中 `CANN_PATH` 的默认值为 `/home/developer/Ascend/cann-9.0.0`，请替换为你的实际 CANN 安装目录。

---
## 7. 编译与运行

执行以下命令编译可执行文件：

In [ ]:
!cd Sources/01.04 && mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/01.04/build/ && \
cmake .. && \
make

再执行以下代码，进行算子的实际运行：

In [ ]:
!cd Sources/01.04/build/ && ./minimal_demo

---
## 课后实践

本节共有 **7 处填空**，分布在 `SetTilingData`（4 处）和 `CheckTilingSpace`（3 处）中。

**SetTilingData 填空（4 处）—— 重点理解 M/K/N 维度映射与前向卷积的差异：**

- `[QUIZ 1/4]` singleCoreM：反向传播的 M 维度 = **CO**（前向中 M = HO×WO）
- `[QUIZ 2/4]` singleCoreK：反向传播的 K 维度 = **HO × WO**（前向中 K = CI×KH×KW）
- `[QUIZ 3/4]` nL0：N 维度对齐到 16 = **AlignUp(CI×KH×KW, 16)**
- `[QUIZ 4/4]` aL1SpaceSize：gradOutput NZ 格式 L1 空间大小

**CheckTilingSpace 填空（3 处）—— 重点理解从 tiling 结构体直接读取已计算值：**

- `[QUIZ 1/3]` al1Bytes/bl1Bytes：从 tiling 读取 L1 空间大小
- `[QUIZ 2/3]` mAlign/kAlign/nAlign：从 tiling 读取 Mmad 对齐维度（`t->mStep`、`t->kL0`、`t->nL0`）
- `[QUIZ 3/5]` cl0Bytes：L0C 字节数 = **mAlign × nAlign × fp32**

In [ ]:
# 在此处填写你的答案，然后运行本单元格验证
# 参考 SetTilingData 和 CheckTilingSpace 中的 [QUIZ] 注释

# [QUIZ 1/4] singleCoreM = ?
CO = 16
singleCoreM = None  # TODO: CO

# [QUIZ 2/4] singleCoreK = ?
HO = 4; WO = 4
singleCoreK = None  # TODO: HO * WO

# [QUIZ 3/4] nL0 = ?
CI = 16; KH = 3; KW = 3
nL0 = None  # TODO: ((CI * KH * KW + 15) // 16) * 16

# [QUIZ 4/4] aL1SpaceSize = ?
aL1SpaceSize = None  # TODO: ((HO * WO + 15) // 16) * 16 * CO * 2

# [QUIZ 1/3] al1Bytes, bl1Bytes from tiling struct
al1Bytes = None   # TODO: t->aL1SpaceSize
bl1Bytes = None   # TODO: t->bL1SpaceSize * 2  (fp16)

# [QUIZ 2/3] mAlign, kAlign, nAlign from tiling struct
mAlign = None   # TODO: t->mStep
kAlign = None   # TODO: t->kL0
nAlign = None   # TODO: t->nL0
al0Bytes = None  # TODO: mAlign * kAlign * 2  (fp16)
bl0Bytes = None  # TODO: kAlign * nAlign * 2  (fp16)

# [QUIZ 3/3] cl0Bytes = ?
cl0Bytes = None  # TODO: mAlign * nAlign * 4  (fp32, NOT 2!)

print("Your answers:")
print(f"  singleCoreM={singleCoreM}, singleCoreK={singleCoreK}, nL0={nL0}, aL1SpaceSize={aL1SpaceSize}")
print(f"  al1Bytes={al1Bytes}, bl1Bytes={bl1Bytes}")
print(f"  mAlign={mAlign}, kAlign={kAlign}, nAlign={nAlign}")
print(f"  al0Bytes={al0Bytes}, bl0Bytes={bl0Bytes}, cl0Bytes={cl0Bytes}")

完成填写后，执行以下命令查看答案：

In [ ]:
!cat ./answer/01.04_answer.txt